# Brain Age Prediction with Stacked Models

This notebook trains brain age prediction models using multiple MRI-derived metrics,
compares different approaches, and generates feature importance maps.

## Workflow
1. Load and preprocess data (GM, WM, CSF volumes; diffusion metrics)
2. Compute post-stratification weights for population-representative estimates
3. Train per-metric Ridge regression models with bias correction
4. Build stacked ensemble (parcel-wise predictions + meta-learner)
5. Evaluate model performance and generate visualizations
6. Compute feature importance (SHAP + permutation)
7. Export predictions for downstream BAG analysis

In [1]:
# Core imports
import numpy as np
import pandas as pd
from pathlib import Path

# ML imports
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.inspection import permutation_importance
import shap
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Neuroaging utilities
from neuroaging.utils import (
    setup_plotting,
    savefig_nice,
    COL_WEIGHTED,
    COL_RAW,
    COL_REF,
    CMAP_WEIGHTED,
    METRIC_LABELS,
    load_metric_data,
    prep_metric_matrices,
    compute_joint_poststrat_weights,
    corrected_cross_val_predict,
    beheshti_bias_correction,
)

/home/groot/Projects/neuroaging/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-02 19:17:33.509 | INFO     | neuroaging.config:<module>:11 - PROJ_ROOT path is: /home/groot/Projects/neuroaging


In [2]:
# Setup visualization
setup_plotting(font_path="/home/galkepler/.fonts/calibri-regular.ttf")

## 1. Configuration

In [3]:
# Paths
# DATA_DIR = Path("/media/storage/phd/neuroaging/data")
# OUTPUT_DIR = Path("/media/storage/phd/neuroaging/figures/revision/fig4")
DATA_DIR = Path("/media/groot/Minerva/phd/neuroaging/data")
OUTPUT_DIR = Path("/media/groot/Minerva/phd/neuroaging/figures/revision/fig4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Atlas configuration
ATLAS = "schaefer2018tian2020_400_7"
REGION_COL = "index"

# Metrics to analyze
METRICS = ["gm_vol", "wm_vol", "csf_vol", "adc", "fa", "ad", "rd"]
BAD_SUBJECTS = ["IN120120"]

# Covariates per metric type
COV_NAMES = {m: ["sex"] if "vol" not in m else ["sex", "tiv"] for m in METRICS}

# Model settings
ALPHAS = np.logspace(-3, 4, 30)
CV_SPLITS = 5
USE_WEIGHTS = False
APPLY_CORRECTION = True

## 2. Load Data

In [4]:
# Load atlas parcellation info
parcels = pd.read_csv(
    DATA_DIR / "external" / "atlases" / ATLAS / "parcels.csv", index_col=0
)
nifti_matlab = DATA_DIR / "external" / "atlases" / ATLAS / "atlas_matlab.nii"

# Load population data for weighting
israel_population = pd.read_csv(DATA_DIR / "processed" / "israel_population.csv")

# Load metric data
data = load_metric_data(
    DATA_DIR, METRICS, bad_subjects=BAD_SUBJECTS, distribution_metric="qfmean"
)

print(f"Loaded {len(METRICS)} metrics")
print(f"Sample sizes: {[len(data[m]) for m in METRICS]}")

/home/groot/Projects/neuroaging/neuroaging/utils/data.py:40: DtypeWarning: Columns (1,7,61,67,71,72,73,74,75,77,78,79,80,85,86,87,88,89,90,92,106,108,109,114,115,116,117,118,125,126,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/home/groot/Projects/neuroaging/neuroaging/utils/data.py:40: DtypeWarning: Columns (1,7,61,67,71,72,73,74,75,77,78,79,80,85,86,87,88,89,90,92,106,108,109,114,115,116,117,118,125,126,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/home/groot/Projects/neuroaging/neuroaging/utils/data.py:40: DtypeWarning: Columns (1,7,61,67,71,72,73,74,75,77,78,79,80,85,86,87,88,89,90,92,106,108,109,114,115,116,117,118,125,126,127) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/home/groot/Projects/neuroaging/neuroaging/utils/data.py:40: DtypeWarning: Columns (16,22,23,26,32,36,37,38,39,40,42,43,44,45,50,51,52,53,54,55,57,71,73,74,7

Loaded 7 metrics
Sample sizes: [1269838, 1269838, 1269838, 1269384, 1269384, 1269384, 1269384]


## 3. Compute Post-stratification Weights

In [5]:
for metric in METRICS:
    data[metric]["poststrat_weight"], _ = compute_joint_poststrat_weights(
        data[metric],
        israel_population,
        age_col="age_at_scan",
        sex_col="sex",
        return_bin_table=True,
        cap=10,
    )

## 4. Prepare Feature Matrices

In [6]:
X_dict, y, w, cov = prep_metric_matrices(data, region_col=REGION_COL)

print(f"Common subjects: {len(y)}")
print(f"Age range: {y.min():.1f} - {y.max():.1f} years")
print(f"Features per metric: {[X_dict[m].shape[1] for m in METRICS]}")

Common subjects: 2796
Age range: 18.0 - 87.3 years
Features per metric: [454, 454, 454, 454, 454, 454, 454]


## 5. Define Model Pipeline

In [7]:
outer_cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=1)

pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("estimator", Ridge()),
])

param_grid = {"estimator__alpha": ALPHAS}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

## 6. Train Per-Metric Models

In [8]:
predictions = {}
perf_rows = []

for metric, X in X_dict.items():
    # Add covariates
    covariates = COV_NAMES[metric]
    X_model = np.hstack([X, cov[covariates].to_numpy()])

    # Fit model
    fit_params = {"estimator__sample_weight": w if USE_WEIGHTS else None}
    grid.fit(X_model, y, **fit_params)
    model = grid.best_estimator_

    # Cross-validated predictions with bias correction
    correction_func = beheshti_bias_correction if APPLY_CORRECTION else None
    y_pred = corrected_cross_val_predict(
        estimator=model,
        X=X_model,
        y=y.values,
        cv=outer_cv,
        params=fit_params,
        correction_func=correction_func,
    )

    # Store predictions
    pred_df = cov.copy()
    pred_df["True"] = y
    pred_df["Predicted"] = y_pred
    pred_df["corrected_residuals"] = y_pred - y
    predictions[metric] = pred_df

    # Fold-level MAEs on the same bias-corrected predictions used for MAE below
    # (for the errorbars/dots plotted in Fig 6A)
    fold_maes = np.array([
        mean_absolute_error(y.values[test_idx], y_pred[test_idx])
        for _, test_idx in outer_cv.split(y)
    ])

    # Raw (uncorrected) OOF predictions on the same folds, used only for the
    # significance test: the Beheshti correction plugs the test fold's own
    # true age into the correction formula, which shrinks between-model MAE
    # differences regardless of real feature-set quality. Testing on the raw,
    # uncorrected fold scores avoids that circularity.
    y_pred_raw = cross_val_predict(
        estimator=model,
        X=X_model,
        y=y.values,
        cv=outer_cv,
        params=fit_params,
    )
    fold_maes_raw = np.array([
        mean_absolute_error(y.values[test_idx], y_pred_raw[test_idx])
        for _, test_idx in outer_cv.split(y)
    ])

    # Record performance
    perf_rows.append({
        "metric": metric,
        "model": model,
        "fold_maes": fold_maes,
        "fold_maes_raw": fold_maes_raw,
        "R2": r2_score(y, y_pred),
        "R2_weighted": r2_score(y, y_pred, sample_weight=w),
        "MAE": mean_absolute_error(y, y_pred),
        "MAE_weighted": mean_absolute_error(y, y_pred, sample_weight=w),
        "RMSE": root_mean_squared_error(y, y_pred),
        "RMSE_weighted": root_mean_squared_error(y, y_pred, sample_weight=w),
    })

    print(f"{metric}: MAE={perf_rows[-1]['MAE']:.2f}, R²={perf_rows[-1]['R2']:.3f}")

gm_vol: MAE=3.77, R²=0.789
wm_vol: MAE=3.92, R²=0.769
csf_vol: MAE=3.71, R²=0.779
adc: MAE=3.64, R²=0.786
fa: MAE=3.54, R²=0.802
ad: MAE=3.61, R²=0.787
rd: MAE=3.63, R²=0.787


## 7. Build Stacked Model (Parcel-wise + Meta-learner)

In [9]:
# Train parcel-wise base learners
stacked_models = parcels.copy()
predictions["base_stacked"] = {}

for i, row in parcels.iterrows():
    # Build design matrix for parcel i (all metrics combined)
    X_roi = np.hstack([X_dict[m][:, [i]] for m in METRICS])

    # Fit model
    fit_params = {"estimator__sample_weight": w if USE_WEIGHTS else None}
    grid.fit(X_roi, y, **fit_params)
    model = grid.best_estimator_

    # Get OOF predictions (no correction at base level)
    y_pred = cross_val_predict(
        estimator=model,
        X=X_roi,
        y=y.values,
        cv=outer_cv,
        params=fit_params,
    )

    # Store predictions
    pred_df = cov.copy()
    pred_df["True"] = y
    pred_df["Predicted"] = y_pred
    pred_df["corrected_residuals"] = y_pred - y
    predictions["base_stacked"][i] = pred_df

    # Store metrics
    stacked_models.loc[i, "MAE"] = mean_absolute_error(y, y_pred)
    stacked_models.loc[i, "R2"] = r2_score(y, y_pred)

print(f"Trained {len(parcels)} parcel-wise models")
print(f"Mean parcel MAE: {stacked_models['MAE'].mean():.2f}")

Trained 454 parcel-wise models
Mean parcel MAE: 6.69


In [10]:
# Build meta-feature matrix and train meta-learner
X_cov = cov[COV_NAMES["gm_vol"]].to_numpy()
X_stacked = np.vstack([
    predictions["base_stacked"][i]["Predicted"].to_numpy()
    for i in predictions["base_stacked"]
]).T
X_stacked = np.hstack([X_stacked, X_cov])

# Fit meta-learner
fit_params = {"estimator__sample_weight": w if USE_WEIGHTS else None}
grid.fit(X_stacked, y, **fit_params)
meta_model = grid.best_estimator_

# Get bias-corrected predictions
correction_func = beheshti_bias_correction if APPLY_CORRECTION else None
y_pred_stacked = corrected_cross_val_predict(
    estimator=meta_model,
    X=X_stacked,
    y=y.values,
    cv=outer_cv,
    params=fit_params,
    correction_func=correction_func,
)

# Store stacked predictions
pred_df = cov.copy()
pred_df["True"] = y
pred_df["Predicted"] = y_pred_stacked
pred_df["corrected_residuals"] = y_pred_stacked - y
predictions["stacked"] = pred_df

# Fold-level MAEs on the same bias-corrected predictions used for MAE below
# (for the errorbars/dots plotted in Fig 6A)
fold_maes_stacked = np.array([
    mean_absolute_error(y.values[test_idx], y_pred_stacked[test_idx])
    for _, test_idx in outer_cv.split(y)
])

# Raw (uncorrected) OOF predictions on the same folds, used only for the
# significance test (see cell above for why: Beheshti correction uses the
# test fold's own true age, which shrinks between-model differences)
y_pred_stacked_raw = cross_val_predict(
    estimator=meta_model,
    X=X_stacked,
    y=y.values,
    cv=outer_cv,
    params=fit_params,
)
fold_maes_stacked_raw = np.array([
    mean_absolute_error(y.values[test_idx], y_pred_stacked_raw[test_idx])
    for _, test_idx in outer_cv.split(y)
])

# Add to performance table
perf_rows.append({
    "metric": "stacked",
    "model": meta_model,
    "fold_maes": fold_maes_stacked,
    "fold_maes_raw": fold_maes_stacked_raw,
    "R2": r2_score(y, y_pred_stacked),
    "R2_weighted": r2_score(y, y_pred_stacked, sample_weight=w),
    "MAE": mean_absolute_error(y, y_pred_stacked),
    "MAE_weighted": mean_absolute_error(y, y_pred_stacked, sample_weight=w),
    "RMSE": root_mean_squared_error(y, y_pred_stacked),
    "RMSE_weighted": root_mean_squared_error(y, y_pred_stacked, sample_weight=w),
})

print(f"Stacked model: MAE={perf_rows[-1]['MAE']:.2f}, R²={perf_rows[-1]['R2']:.3f}")

Stacked model: MAE=3.45, R²=0.811


## 8. Performance Comparison

In [11]:
perf_df = pd.DataFrame(perf_rows).set_index("metric").sort_values("MAE")
perf_df["is_best"] = perf_df["MAE"] == perf_df["MAE"].min()
display(perf_df[["MAE", "MAE_weighted", "R2", "R2_weighted", "RMSE"]])

,MAE,MAE_weighted,R2,R2_weighted,RMSE
metric,,,,,
stacked,3.453467,4.206518,0.811313,0.900088,4.483723
fa,3.543958,4.322253,0.802490,0.892967,4.587356
ad,3.613889,4.389477,0.787044,0.892110,4.763356
rd,3.632136,4.556756,0.786638,0.880099,4.767888
adc,3.636259,4.524647,0.786075,0.885090,4.774183
csf_vol,3.710099,4.597449,0.779288,0.878704,4.849321
gm_vol,3.774848,4.087720,0.788801,0.913418,4.743659
wm_vol,3.918588,4.256087,0.768959,0.902714,4.961495


In [12]:
# # Performance barplot with significance annotations (Fig 6A)
# import matplotlib as mpl
# from scipy import stats as sstats

# mpl.rcParams.update({"axes.labelsize": 35, "ytick.labelsize": 25, "xtick.labelsize": 25})

# # Per-fold MAE (outer_cv's folds only depend on n_samples/random_state, so they're
# # identical to the folds each model was actually evaluated on)
# fold_rows = []
# for metric in perf_df.index:
#     pred_df = predictions[metric]
#     for fold_id, (_, test_idx) in enumerate(outer_cv.split(y)):
#         fold_rows.append({
#             "metric": metric,
#             "fold": fold_id,
#             "MAE": mean_absolute_error(pred_df["True"].iloc[test_idx], pred_df["Predicted"].iloc[test_idx]),
#         })
# fold_df = pd.DataFrame(fold_rows)
# fold_mae_wide = fold_df.pivot(index="fold", columns="metric", values="MAE")
# perf_df["fold_maes"] = [fold_mae_wide[m].to_numpy() for m in perf_df.index]

# # Corrected resampled paired t-test vs. stacked model (Nadeau & Bengio, 2003):
# # accounts for fold non-independence, unlike a plain paired t-test on CV scores.
# n_test = np.mean([len(test_idx) for _, test_idx in outer_cv.split(y)])
# n_train = len(y) - n_test
# stacked_folds = fold_mae_wide["stacked"].to_numpy()

# stats_rows = []
# for metric in perf_df.index:
#     if metric == "stacked":
#         continue
#     diff = fold_mae_wide[metric].to_numpy() - stacked_folds
#     var_diff = diff.var(ddof=1)
#     denom = np.sqrt((1 / CV_SPLITS + n_test / n_train) * var_diff)
#     t_stat = diff.mean() / denom if denom > 0 else np.inf
#     p_value = 2 * sstats.t.sf(np.abs(t_stat), df=CV_SPLITS - 1)
#     sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
#     stats_rows.append({"metric": metric, "t_stat": t_stat, "p_value": p_value, "sig": sig})

# stats_df = pd.DataFrame(stats_rows).set_index("metric")
# display(stats_df)


In [13]:
def corrected_resampled_ttest(mae_a, mae_b, n, k):
    """Nadeau & Bengio (2003) corrected resampled t-test for k-fold CV.

    Adjusts the variance estimate to account for train/test overlap between
    folds. Tests H0: mean(mae_a) == mean(mae_b).

    Parameters
    ----------
    mae_a, mae_b : array of shape (k,)  per-fold MAEs for each model
    n            : total number of samples
    k            : number of CV folds

    Returns
    -------
    t_stat, p_value (two-tailed)
    """
    d = mae_a - mae_b          # positive → A is worse than B
    mean_d = d.mean()
    var_d = d.var(ddof=1)
    n_test = n // k
    n_train = n - n_test
    corrected_var = (1 / k + n_test / n_train) * var_d
    t_stat = mean_d / np.sqrt(corrected_var)
    p_val = 2 * stats.t.sf(np.abs(t_stat), df=k - 1)
    return t_stat, p_val


# Tested on raw (uncorrected) fold MAEs, not the Beheshti-corrected ones:
# the correction plugs each test fold's own true age into the bias term,
# which shrinks between-model differences regardless of real feature-set
# quality and would make this test answer the wrong question.
n_subjects = len(y)
stacked_folds = perf_df.loc["stacked", "fold_maes_raw"]

stat_rows = []
for metric in METRICS:
    metric_folds = perf_df.loc[metric, "fold_maes_raw"]
    t, p = corrected_resampled_ttest(metric_folds, stacked_folds, n_subjects, CV_SPLITS)
    stat_rows.append({"metric": metric, "t_stat": t, "p_value": p})

stats_df = pd.DataFrame(stat_rows).set_index("metric")
stats_df["sig"] = pd.cut(
    stats_df["p_value"],
    bins=[0, 0.001, 0.01, 0.05, 1.0],
    labels=["***", "**", "*", "ns"],
)
display(stats_df.round(4))

,t_stat,p_value,sig
metric,,,
gm_vol,9.9612,0.0006,***
wm_vol,14.2881,0.0001,***
csf_vol,11.9857,0.0003,***
adc,6.7170,0.0026,**
fa,6.6696,0.0026,**
ad,7.6886,0.0015,**
rd,6.7244,0.0025,**


In [25]:
COL_WEIGHTED = "#7F0099CE"  # green - Set2[0]
COL_RAW = "#ffb300"  # orange - Set2[1]

In [30]:
# Performance barplot with significance annotations (Fig 6A)
import matplotlib as mpl
mpl.rcParams.update({"axes.labelsize": 35, "ytick.labelsize": 25, "xtick.labelsize": 25})

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    x="metric",
    y="MAE",
    data=perf_df.reset_index(),
    ax=ax,
    hue="is_best",
    legend=False,
    alpha=0.8,
    palette=[COL_RAW, COL_WEIGHTED],
)

# add errorbars (CI across fold)
for i, metric in enumerate(perf_df.index):
    fold_maes = perf_df.loc[metric, "fold_maes"]
    mae_mean = perf_df.loc[metric, "MAE"]
    mae_se = fold_maes.std(ddof=1) / np.sqrt(len(fold_maes))
    ax.errorbar(i, mae_mean, yerr=mae_se * 1.96, fmt="none", c="black", capsize=5)

# overlay individual CV fold scores
rng = np.random.default_rng(0)
for i, metric in enumerate(perf_df.index):
    fold_maes = perf_df.loc[metric, "fold_maes"]
    # jitter = rng.uniform(-0.08, 0.08, size=len(fold_maes))
    jitter = rng.uniform(0, 0, size=len(fold_maes))
    ax.scatter(i + jitter, fold_maes, color="black", alpha=0.6, s=30, zorder=3)

ax.set_xticks(range(len(perf_df)))
ax.set_xticklabels([METRIC_LABELS.get(m, m) for m in perf_df.index], rotation=45)
ax.set_ylabel("MAE (years)", fontsize=35)
ax.set_xlabel("")
ax.set_title("(A) Performance Comparison", fontsize=40, pad=20)
ax.set_ylim(3, 4.4)

# Significance markers above each unimodal bar (vs. stacked)
for bar_idx, metric in enumerate(perf_df.index):
    if metric == "stacked" or metric not in stats_df.index:
        continue
    sig = str(stats_df.loc[metric, "sig"])
    # place text above the error bar (using the bar height + a small offset)
    mae_mean = perf_df.loc[metric, "MAE"]
    mae_se = perf_df.loc[metric, "fold_maes"].std(ddof=1) / np.sqrt(len(perf_df.loc[metric, "fold_maes"]))
    bar_top = mae_mean + mae_se * 1.96

    ax.text(
        bar_idx, bar_top + 0.04, sig,
        ha="center", va="bottom",
        fontsize=16, fontweight="bold",
        color="black",
    )

ax.text(
    0.77, 0.994,
    "* p<0.05  ** p<0.01  *** p<0.001\n(corrected resampled t-test vs. stacked)",
    transform=ax.transAxes,
    ha="center", va="top",
    fontsize=15, color="black",
)


plt.tight_layout()
savefig_nice(fig, OUTPUT_DIR / "model_performance.png", dpi=300)

In [15]:
perf_df

,model,fold_maes,fold_maes_raw,R2,R2_weighted,MAE,MAE_weighted,RMSE,RMSE_weighted,is_best
metric,,,,,,,,,,
stacked,"(RobustScaler(), Ridge(alpha=np.float64(356.22...","[3.333927702856176, 3.5574788022059027, 3.2616...","[3.9426486301923336, 3.908255671364615, 3.7139...",0.811313,0.900088,3.453467,4.206518,4.483723,5.493638,True
fa,"(RobustScaler(), Ridge(alpha=np.float64(621.01...","[3.7310150384748715, 3.4669921982924246, 3.385...","[4.73569423340004, 4.379160236606987, 4.258000...",0.802490,0.892967,3.543958,4.322253,4.587356,5.686026,False
ad,"(RobustScaler(), Ridge(alpha=np.float64(621.01...","[3.7666918745510265, 3.588503409113412, 3.3361...","[4.889170780018495, 4.631193995779498, 4.34292...",0.787044,0.892110,3.613889,4.389477,4.763356,5.708738,False
rd,"(RobustScaler(), Ridge(alpha=np.float64(621.01...","[3.824372177112063, 3.6155632169721055, 3.2012...","[4.87356517893608, 4.725880585194143, 4.222415...",0.786638,0.880099,3.632136,4.556756,4.767888,6.018129,False
adc,"(RobustScaler(), Ridge(alpha=np.float64(621.01...","[3.785133026931625, 3.5867698065358455, 3.2412...","[4.869189035000338, 4.733428755108674, 4.23512...",0.786075,0.885090,3.636259,4.524647,4.774183,5.891531,False
csf_vol,"(RobustScaler(), Ridge(alpha=np.float64(356.22...","[3.771082189001205, 3.8441022593332166, 3.6263...","[5.0406648138626675, 4.75148083823785, 4.64904...",0.779288,0.878704,3.710099,4.597449,4.849321,6.053027,False
gm_vol,"(RobustScaler(), Ridge(alpha=np.float64(117.21...","[3.677681605721316, 3.9022045285934936, 3.6725...","[4.647178481660717, 4.452708315426567, 4.46732...",0.788801,0.913418,3.774848,4.087720,4.743659,5.114033,False
wm_vol,"(RobustScaler(), Ridge(alpha=np.float64(204.33...","[3.9477438915460175, 4.238534943597258, 3.7678...","[5.158997604527607, 5.283526335554016, 5.03625...",0.768959,0.902714,3.918588,4.256087,4.961495,5.420941,False


In [16]:
fold_maes

array([3.94774389, 4.23853494, 3.76780505, 3.64612287, 3.99268039])

In [20]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    x="metric",
    y="MAE",
    data=perf_df.reset_index(),
    ax=ax,
    hue="is_best",
    legend=False,
    alpha=0.8,
    palette=[COL_RAW, COL_WEIGHTED],
)

# add errorbars (CI across fold)
for i, metric in enumerate(perf_df.index):
    fold_maes = perf_df.loc[metric, "fold_maes"]
    mae_mean = perf_df.loc[metric, "MAE"]
    mae_se = fold_maes.std(ddof=1) / np.sqrt(len(fold_maes))
    ax.errorbar(i, mae_mean, yerr=mae_se * 1.96, fmt="none", c="black", capsize=5)

ax.set_xticks(range(len(perf_df)))
ax.set_xticklabels([METRIC_LABELS.get(m, m) for m in perf_df.index], rotation=45)
ax.set_ylabel("MAE (years)", fontsize=35)
ax.set_xlabel("")
ax.set_title("(A) Performance Comparison", fontsize=40, pad=20)
ax.set_ylim(3, 4.3)

# Significance markers above each unimodal bar (vs. stacked)
for bar_idx, metric in enumerate(perf_df.index):
    if metric == "stacked" or metric not in stats_df.index:
        continue
    sig = str(stats_df.loc[metric, "sig"])
    # place text above the error bar (using the bar height + a small offset)
    mae_mean = perf_df.loc[metric, "MAE"]
    mae_se = perf_df.loc[metric, "fold_maes"].std(ddof=1) / np.sqrt(len(perf_df.loc[metric, "fold_maes"]))
    bar_top = mae_mean + mae_se * 1.96

    ax.text(
        bar_idx, bar_top + 0.02, sig,
        ha="center", va="bottom",
        fontsize=16, fontweight="bold",
        color="black",
    )

ax.text(
    0.77, 0.994,
    "* p<0.05  ** p<0.01  *** p<0.001\n(corrected resampled t-test vs. stacked)",
    transform=ax.transAxes,
    ha="center", va="top",
    fontsize=15, color="black",
)


plt.tight_layout()
savefig_nice(fig, OUTPUT_DIR / "model_performance.png", dpi=300)

## 9. Predicted vs True Age Visualization

In [18]:
best_metric = perf_df.index[0]
y_true = predictions[best_metric]["True"].to_numpy()
y_pred = predictions[best_metric]["Predicted"].to_numpy()
residuals = y_pred - y_true

r2 = r2_score(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(20, 8), width_ratios=[2, 1])

# Panel A: Predicted vs True Age
ax = axes[0]
sns.scatterplot(x=y_true, y=y_pred, ax=ax, color=COL_REF, alpha=0.7, s=w * 10)
ax.plot([10, 100], [10, 100], "--", c="red", lw=5, alpha=0.5, label="Perfect Prediction")

# Regression line with CI
lin_m = sm.OLS(y_pred, sm.add_constant(y_true)).fit()
x_vals = np.linspace(y_true.min(), y_true.max(), 300)[:, np.newaxis]
y_vals = lin_m.predict(sm.add_constant(x_vals))
ax.plot(x_vals, y_vals, lw=5, label=f"Model ($R^2$={r2:.2f}, MAE={mae:.2f})", color=COL_WEIGHTED)

preds = lin_m.get_prediction(sm.add_constant(x_vals)).summary_frame(alpha=0.05)
ax.fill_between(x_vals.flatten(), preds["mean_ci_lower"], preds["mean_ci_upper"],
                color=COL_WEIGHTED, alpha=0.3)

ax.set_xlabel("True Age")
ax.set_ylabel("Predicted Age")
ax.set_title("(B) Predicted vs. True Age", fontsize=40, pad=20)
ax.set_aspect("equal", "box")
ax.legend(frameon=False, loc="lower right", fontsize=20)

# Panel B: BAG vs Age
ax = axes[1]
sns.scatterplot(x=y_true, y=residuals, ax=ax, color=COL_REF, alpha=0.7, s=w * 10)
ax.axhline(0, color="red", linestyle="--")

lin_m = sm.OLS(residuals, sm.add_constant(y_true)).fit()
y_vals = lin_m.predict(sm.add_constant(x_vals))
ax.plot(x_vals, y_vals, lw=5, label=f"$R^2$={lin_m.rsquared:.2f}", color=COL_WEIGHTED)
# add CI for regression line
preds = lin_m.get_prediction(sm.add_constant(x_vals)).summary_frame(alpha=0.05)
ax.fill_between(x_vals.flatten(), preds["mean_ci_lower"], preds["mean_ci_upper"],
                color=COL_WEIGHTED, alpha=0.3)

ax.set_xlabel("Age (years)")
ax.set_ylabel("BAG")
ax.set_title("(C) BAG vs. True Age", fontsize=40, pad=20)
ax.set_ylim(-30, 30)
ax.legend(frameon=False, fontsize=20)

plt.tight_layout()
savefig_nice(fig, OUTPUT_DIR / "prediction_scatter.png", dpi=300)

In [19]:
raise

RuntimeError: No active exception to reraise

## 10. Feature Importance (SHAP + Permutation)

In [ ]:
best_model = perf_df.loc[best_metric, "model"]

# SHAP values
X_transformed = best_model[:-1].transform(X_stacked)
explainer = shap.LinearExplainer(best_model.named_steps["estimator"], X_transformed)
shap_values = explainer(X_transformed)

# Permutation importance
perm_result = permutation_importance(
    estimator=best_model,
    X=X_stacked,
    y=y,
    scoring="neg_mean_squared_error",
    n_repeats=100,
    random_state=42,
)

# Combine importance measures
shap_imp = parcels.copy()
shap_imp["importance"] = np.abs(shap_values.values).mean(axis=0)[:len(parcels)]
shap_imp["importance_scaled"] = (
    (shap_imp["importance"] - shap_imp["importance"].min()) /
    (shap_imp["importance"].max() - shap_imp["importance"].min())
)
shap_imp["permutation_importance"] = perm_result.importances_mean[:len(parcels)]
shap_imp.loc[shap_imp["permutation_importance"] < 0, "permutation_importance"] = 0
shap_imp["permutation_scaled"] = (
    (shap_imp["permutation_importance"] - shap_imp["permutation_importance"].min()) /
    (shap_imp["permutation_importance"].max() - shap_imp["permutation_importance"].min())
)

display(shap_imp.sort_values("permutation_scaled", ascending=False).head(10))

,index,name,base_name,Label Name,network,component,hemisphere,importance,importance_scaled,permutation_importance,permutation_scaled
441,442,PUT-VP-lh,PUT-VP,"Putamen, ventro-posterior part",subcortex,Putamen,L,0.611647,1.000000,1.594409,1.000000
437,438,THA-DAm-lh,THA-DAm,"Thalamus, medial dorso-anterior part",subcortex,Thalamus,L,0.471173,0.770308,1.067556,0.669562
438,439,THA-DAl-lh,THA-DAl,"Thalamus, lateral dorso-anterior part",subcortex,Thalamus,L,0.462023,0.755346,0.969705,0.608190
413,414,PUT-DA-rh,PUT-DA,"Putamen, dorso-anterior part",subcortex,Putamen,R,0.458122,0.748966,0.894068,0.560752
411,412,THA-DAl-rh,THA-DAl,"Thalamus, lateral dorso-anterior part",subcortex,Thalamus,R,0.391138,0.639440,0.706447,0.443078
435,436,THA-VPl-lh,THA-VPl,"Thalamus, lateral ventro-posterior part",subcortex,Thalamus,L,0.422052,0.689987,0.692903,0.434583
309,310,7Networks_RH_SalVentAttn_PFCl_1,7networks_rh_salventattn_pfcl,7Networks_RH_SalVentAttn_PFCl,salience / ventral attention,lateral prefrontal cortex,R,0.388292,0.634786,0.601861,0.377482
232,233,7Networks_RH_SomMot_3,7networks_rh_sommot,7Networks_RH_SomMot,somatomotor,somatomotor,R,0.363735,0.594633,0.566341,0.355204
218,219,7Networks_RH_Vis_19,7networks_rh_vis,7Networks_RH_Vis,visual,visual,R,0.367699,0.601115,0.542074,0.339984
245,246,7Networks_RH_SomMot_16,7networks_rh_sommot,7Networks_RH_SomMot,somatomotor,somatomotor,R,0.362293,0.592274,0.496809,0.311594


: 

## 11. Brain Surface Visualization

In [ ]:
import nibabel as nib
from neuromaps.datasets import fetch_fslr
from brainspace.datasets import load_parcellation
from surfplot import Plot

# Load surfaces and parcellation
surfaces = fetch_fslr()
lh, rh = surfaces["inflated"]
lh_parc, rh_parc = load_parcellation("schaefer")

# Map importance values to surface
vmin, vmax = 0, 0.1
value_column = "permutation_importance"

value_map_lh = {}
value_map_rh = {}

for i, row in shap_imp.iterrows():
    label = row[REGION_COL]
    hemi = row["hemisphere"]
    value = row[value_column] if row[value_column] >= 0 else np.nan

    if label <= int(ATLAS.split("_")[1]):  # Cortical parcels
        if hemi == "L":
            value_map_lh[label] = value
        elif hemi == "R":
            value_map_rh[label] = value

# Apply mapping
data_lh = np.vectorize(lambda x: value_map_lh.get(x, np.nan))(lh_parc)
data_rh = np.vectorize(lambda x: value_map_rh.get(x, np.nan))(rh_parc)

# Create surface plot
p = Plot(lh, rh, views=["lateral", "medial"], zoom=1.2)
p.add_layer({"left": data_lh, "right": data_rh}, cmap=CMAP_WEIGHTED,
            color_range=(vmin, vmax), cbar=True, cbar_label="Importance")

fig = p.build()
savefig_nice(fig, OUTPUT_DIR / "feature_importance_surface.png", dpi=300)

## 12. Export Predictions for BAG Analysis

In [ ]:
# Compile all predictions into single DataFrame
predictions_df = pd.DataFrame()

for metric in predictions:
    if metric == "base_stacked":
        for i, row in parcels.iterrows():
            cur_df = predictions[metric][i][["True", "Predicted"]].copy()
            cur_df = cur_df.rename(columns={
                "True": f"base_{i}_age",
                "Predicted": f"base_{i}_predicted"
            })
            cur_df[f"base_{i}_BAG"] = cur_df[f"base_{i}_predicted"] - cur_df[f"base_{i}_age"]
            predictions_df = pd.concat([predictions_df, cur_df], axis=1)
    else:
        cur_df = predictions[metric][["True", "Predicted"]].copy()
        cur_df = cur_df.rename(columns={
            "True": f"{metric}_age",
            "Predicted": f"{metric}_predicted"
        })
        cur_df[f"{metric}_BAG"] = cur_df[f"{metric}_predicted"] - cur_df[f"{metric}_age"]
        predictions_df = pd.concat([predictions_df, cur_df], axis=1)

predictions_df.to_csv("BAG_data.csv")
print(f"Saved predictions to BAG_data.csv: {predictions_df.shape}")

Saved predictions to BAG_data.csv: (2796, 1386)
